# 01 — Data Preparation

Download Binance USDT-M full kline data with all 12 columns.

In [ ]:
import torch, sys
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime → T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE  = '/content/drive/MyDrive/yeniBot'
DATA_DIR    = f'{DRIVE_BASE}/data'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
os.makedirs(f'{DATA_DIR}/raw', exist_ok=True)
os.makedirs(f'{DATA_DIR}/processed', exist_ok=True)
os.makedirs(CHECKPT_DIR, exist_ok=True)

In [ ]:
import os, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print('After git pull, use Runtime → Restart session before trusting changed imports.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['data_dir'] = DATA_DIR
cfg['paths']['checkpoint_dir'] = CHECKPT_DIR
cfg

In [ ]:
from yenibot.data import download_full_klines, validate_full_kline_frame

binance = cfg['binance']
raw_dir = f"{DATA_DIR}/raw"
downloads = [(binance['primary_interval'], 'btc_1h'), (binance['htf_interval'], 'btc_4h')]
for intrabar_interval in binance.get('intrabar_intervals', []):
    downloads.append((intrabar_interval, f"btc_{intrabar_interval}"))

for interval, name in downloads:
    df = download_full_klines(
        binance['symbol'],
        interval,
        binance['start_date'],
        binance.get('end_date'),
        base_url=binance['base_url'],
        vision_base_url=binance['vision_base_url'],
        data_source=binance.get('data_source', 'auto'),
        limit=binance['limit'],
        request_sleep_seconds=binance['request_sleep_seconds'],
    )
    df = validate_full_kline_frame(
        df,
        interval,
        max_gap_multiplier=binance['max_gap_multiplier'],
        zero_volume_policy=binance.get('zero_volume_policy', 'error'),
    )
    dropped = df.attrs.get('dropped_zero_volume_rows', 0)
    if dropped:
        print(f'{name}: dropped {dropped} zero-volume/no-trade rows before validation')
    assert 'taker_buy_base_vol' in df.columns and df['taker_buy_base_vol'].abs().sum() > 0
    out = f'{raw_dir}/{name}.parquet'
    df.to_parquet(out, index=False)
    print(name, len(df), df['timestamp'].min(), df['timestamp'].max(), out)